# Part 1 · Where do people want to go?
### OD weights and how pedestrian flows spread through a network

You'll change **who wants to go where** in a small station and watch *where* and *when* crowds form.
Work in pairs: one person clicks, the other writes down predictions. Swap every exercise.


**How to use this notebook**
* Press **▶** on the left of a grey cell (or click it and press **Shift + Enter**) to run it.
* Cells with sliders and boxes are *forms*: change the values on the right, then press ▶ again.
* You never have to edit code. If you're curious, double-click a form to see the code behind it.
* Something broken? **Runtime → Restart session**, then run the first cell again.

## ▶ Step 1: install (run this first, ≈30 s)
This is how you install **any** Python package from GitHub. `pednstream-workshop` is a small
package with this workshop's network and plotting helpers; installing it also installs
**PedNStream** itself. If Colab restarted, run it again.
You can ignore the note *"you may need to restart the kernel"*.

In [ ]:
%pip install -q "pednstream-workshop @ git+https://github.com/WaimenMak/pednstream-workshop.git"

import matplotlib.pyplot as plt
from pednstream_workshop import run, compare, compare_maps, switch
plt.rcParams["figure.dpi"] = 90
print("✅ Ready! Scroll down and continue.")

## 0 · Meet *Station Square*
```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```
* **Two entrances** (green squares): West Gate **A** enters at the Plaza, South Gate **B** enters at the Hall.
* **Two destinations** (red diamonds): the **Platform X** and the **City Exit Y**.
* Corridor **2_3** (Plaza–Hall) is only **2 m** wide.
* A rush of people arrives at both gates, peaking around minute 17. The rush is **the same in every run**.

In [ ]:
#@title What is an OD weight? Try it {display-mode: "form"}
#@markdown OD weights are **relative**. Set two weights for Gate A and see what share of Gate A's people goes where.
weight_A_to_Platform = 3  #@param {type:"slider", min:0, max:10, step:1}
weight_A_to_CityExit = 1  #@param {type:"slider", min:0, max:10, step:1}
total = weight_A_to_Platform + weight_A_to_CityExit
if total == 0:
    print("Both weights are 0. PedNStream then has no preference and falls back to 50% / 50%.")
else:
    pX = weight_A_to_Platform / total
    print(f"Of everyone entering at Gate A:  {pX:.0%} → Platform X,   {1 - pX:.0%} → City Exit Y")
    print("How MANY people enter is set by the gate's demand; the weights only split them.")
    plt.figure(figsize=(6, 1.2)); plt.barh([0], [pX], color="#1565c0"); plt.barh([0], [1 - pX], left=[pX], color="#ef6c00")
    plt.xlim(0, 1); plt.yticks([]); plt.title("Gate A: Platform X (blue) / City Exit Y (orange)"); plt.show()

> **Quick check:** weights 10 & 10 versus 1 & 1. Same result or different? Try it above.

In [ ]:
#@title ▶ Run the baseline (all weights equal) {display-mode: "form"}
base = run()
base.dashboard("Baseline: every gate sends 50% to X and 50% to Y"); plt.show()
display(base.kpis().to_frame("baseline"))

**How to read this**
* **Map**: corridor colour = density (people per m²) at the busiest moment. Above **2** (dashed line) a corridor is *congested* and people slow down sharply.
* **Density per corridor**: when and where crowds build up and clear.
* **Arrivals**: people reaching each destination over time. *Turned away* = people who couldn't even enter because the first corridor was full.
* **time spent inside (person-hours)**: our main "how bad was it" number. **Lower is better.**

---
## Exercise 1 · Everyone to one destination
A train is delayed, and **everyone** heads to the Platform X. Or a concert is on in town and **everyone** heads to the City Exit Y.

In [ ]:
#@title ✋ **Predict first**: Exercise 1 { display-mode: "form" }
#@markdown Which corridor will be worst in each case? Which case is worse overall?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 1 {display-mode: "form"}
all_x = run(od={"0_5": 1, "0_6": 0, "1_5": 1, "1_6": 0})
all_y = run(od={"0_5": 0, "0_6": 1, "1_5": 0, "1_6": 1})
display(compare({"baseline": base, "all to X": all_x, "all to Y": all_y}))
compare_maps({"baseline": base, "all to X": all_x, "all to Y": all_y}); plt.show()
print("How 'all to X' travellers split over their routes:")
display(all_x.route_split())

**Discuss**
1. In *all to X*, people are turned away at **Gate B**. Why B and not A? *(Hint: count B's routes to X in the table.)*
2. The narrow corridor 2_3 is *not* the worst. Where did the crowd go instead, and why?

---
## Exercise 2 · Same totals, different mix ⭐
Both scenarios send **exactly half** of all travellers to X and half to Y. Only *who* goes where differs:

| scenario | Gate A goes to | Gate B goes to |
|---|---|---|
| **crossing** | Platform X (east) | City Exit Y (north) |
| **parallel** | City Exit Y (north) | Platform X (east) |

In [ ]:
#@title ✋ **Predict first**: Exercise 2 { display-mode: "form" }
#@markdown Will the two scenarios give (almost) the same result? If not, which is worse, and where?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 2 {display-mode: "form"}
crossing = run(od={"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1})
parallel = run(od={"0_5": 0, "0_6": 1, "1_5": 1, "1_6": 0})
display(compare({"baseline": base, "crossing": crossing, "parallel": parallel}))
compare_maps({"crossing": crossing, "parallel": parallel}); plt.show()
print("Crossing scenario, corridor by corridor:")
display(crossing.link_table())

**Discuss**
1. Destination totals are identical, yet one scenario costs much more. Explain why in one sentence, using *route overlap* or *counter-flow*.
2. Knowing only "50% go to the Platform", could you have predicted the congestion?

---
## Exercise 3 · Destinations that change over time
A train is announced, and from that moment people become much more likely to head to the Platform.

In [ ]:
#@title ✋ **Predict first**: Exercise 3 { display-mode: "form" }
#@markdown People already walking to the City Exit when the announcement comes: do they keep going or turn around? Does the timing of the announcement matter?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 3 {display-mode: "form"}
announcement_minute = 25  #@param {type:"slider", min:0, max:49, step:1}
times_more_likely_to_go_to_Platform = 6  #@param {type:"slider", min:1, max:10, step:1}
step = int(announcement_minute * 6)       # 1 step = 10 s
k = times_more_likely_to_go_to_Platform
announce = run(od={"0_5": switch(1, k, at_step=step), "0_6": 1,
                   "1_5": switch(1, k, at_step=step), "1_6": 1})
ax = announce.plot_links(["0_2", "2_3", "3_5", "2_4", "4_6"], title=f"Announcement at minute {announcement_minute}")
ax.axvline(announcement_minute, color="k", lw=2, alpha=0.4); plt.show()
display(compare({"baseline": base, f"announced at min {announcement_minute}": announce}))

**Discuss**
1. Look at the vertical line. Do the corridors react with a delay, or all at once?
2. PedNStream applies the **current** OD weights at every junction, so people already inside "re-decide" at their next junction.
   Is that realistic for a loudspeaker announcement? For a departure board visible only at the entrance?
3. Try the announcement at minute **17** (the peak) and at minute **35**. Does *timing* matter as much as *size*?

---
## Exercise 4 · Do people dodge the jam?
People choose between two routes using **distance**, **crowding** and **width**.
**β (beta)** is how strongly people avoid crowds: 0 = they ignore crowding, 10 = they avoid it strongly.

In [ ]:
#@title ✋ **Predict first**: Exercise 4 { display-mode: "form" }
#@markdown For the bad crossing pattern, will people avoiding crowds (high β) fix the problem? Will a wider corridor?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 4 {display-mode: "form"}
scenario = "crossing (A to X, B to Y)"  #@param ["baseline (everyone 50/50)", "all to Platform X", "all to City Exit Y", "crossing (A to X, B to Y)", "parallel (A to Y, B to X)"]
beta_crowd_avoidance = 8  #@param {type:"slider", min:0, max:10, step:0.5}
width_of_corridor_2_3_m = 2  #@param {type:"slider", min:1, max:6, step:0.5}
od = {
    "baseline (everyone 50/50)":        {"0_5": 1, "0_6": 1, "1_5": 1, "1_6": 1},
    "all to Platform X":                {"0_5": 1, "0_6": 0, "1_5": 1, "1_6": 0},
    "all to City Exit Y":               {"0_5": 0, "0_6": 1, "1_5": 0, "1_6": 1},
    "crossing (A to X, B to Y)":        {"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1},
    "parallel (A to Y, B to X)":        {"0_5": 0, "0_6": 1, "1_5": 1, "1_6": 0},
}[scenario]
default = run(od=od)
changed = run(od=od, route_choice={"beta": beta_crowd_avoidance}, links={"2_3": {"width": width_of_corridor_2_3_m}})
label = f"β={beta_crowd_avoidance}, 2_3 = {width_of_corridor_2_3_m} m"
display(compare({"default (β=2, 2_3 = 2 m)": default, label: changed}))
compare_maps({"default": default, label: changed}); plt.show()

**Discuss**
1. Raising β changes the *peak density* more than the *total time*. Why can't smart walkers fix a bad OD pattern on their own?
2. Which helps more here: better behaviour (β), a wider corridor, or a different OD mix (Exercise 2)?

---
## Wrap-up: complete these three sentences with your partner
1. An OD weight controls …
2. Two OD patterns with the same destination totals can behave differently because …
3. When OD weights change over time, the network …

➡ Next: **Part 2 · Playground**, then **Part 3 · Challenge**.